# Comparaison des plans de remboursement de prêts étudiants avec PROC LOAN

## Résumé analytique

Un bureau d'aide financière évalue comment une cohorte de diplômés doit rembourser un solde représentatif de 27 500 $ en comparant quatre structures de remboursement — un plan standard fédéral à taux fixe, un refinancement privé avec des frais de dossier, un prêt à taux variable plafonné (ARM), et un rachat de taux subventionné par l'employeur — à l'aide de `PROC LOAN`. Sur une durée de 120 mois, les quatre offres s'alignent clairement sur la mensualité et l'intérêt total à leurs taux de départ annoncés : le plan standard fédéral coûte le plus d'intérêts (**10 021,22 $** à 6,53 %, mensualité **312,68 $**), le refinancement privé réduit le taux mais ajoute des frais de 412,50 $ (**9 120,20 $** à 5,99 %, mensualité **305,17 $**), et l'ARM au taux affiché le plus bas (**7 099,76 $** à 4,75 %, mensualité **288,33 $**) ainsi que le rachat de taux par l'employeur (**6 700,67 $** à 4,50 %, mensualité **285,01 $**) portent les factures d'intérêts les plus faibles. Un bloc `COMPARE` indique ensuite, pour chaque plan, l'intérêt cumulé, le capital cumulé et le solde restant dû à 36, 60 et 120 mois, montrant à quel point chaque prêt est amorti aux moments où un emprunteur est le plus susceptible de refinancer ou de rembourser par anticipation.

## Sources de données

| Jeu de données | Lignes | Description | Variables clés |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Profils de prêts synthétiques d'une cohorte de diplômés, générés en ligne avec `call streaminit(20260531)` et `rand('uniform')`. Utilisés pour motiver des conditions de prêt réalistes pour la comparaison. | `student_id` (1001-1040), `balance` (capital au moment du diplôme ; ce tirage s'étend de 11 800 $ à 47 750 $, moyenne 30 771 $), `apr` (taux nominal annuel de 4,10 % à 9,10 %, moyenne 6,50 %), `term` (120 ou 180 mois, moyenne 144), `origination_pct` (frais de 1,0 % à 2,0 %, moyenne 1,50 %) |

L'emprunteur représentatif analysé avec `PROC LOAN` (montant 27 500 $, durée de 120 mois, démarrage en juillet 2026) se situe près du milieu inférieur de la distribution des soldes et des taux de cette cohorte ; aucune donnée externe ou réseau n'est utilisée. La cohorte sert à motiver des conditions de prêt plausibles — la comparaison directe est effectuée sur le prêt représentatif unique.

# Comparaison des plans de remboursement de prêts étudiants avec PROC LOAN

Lorsque les étudiants obtiennent leur diplôme, un bureau d'aide financière doit les aider à choisir parmi des offres de remboursement concurrentes. `PROC LOAN` (SAS/ETS) est conçu précisément pour cela : il amortit les prêts à taux fixe, à taux variable (ARM) et à rachat de taux, puis les compare côte à côte sur la mensualité, l'intérêt total et la progression de l'amortissement.

Dans ce notebook, nous allons :

1. Générer une cohorte synthétique de diplômés pour établir des conditions de prêt réalistes.
2. Résumer la cohorte avec `PROC MEANS`.
3. Construire quatre plans de remboursement pour un solde représentatif de 27 500 $ et les comparer avec `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Réexécuter seul le plan fixe recommandé pour confirmer son économie autonome.

Tout s'exécute localement sur des données synthétiques en ligne — aucun fichier externe ni accès réseau.

## Étape 1 — Générer une cohorte synthétique de diplômés

Nous simulons 40 emprunteurs. Chacun tire un solde de capital au moment du diplôme, un taux annuel (APR) faiblement lié à la qualité de crédit, une durée de remboursement standard (10 ou 15 ans), et un pourcentage de frais de dossier. La graine fixe rend les données reproductibles.

In [1]:
DONNÉES borrowers;
   APPELER streaminit(20260531);
   LONGUEUR plan $ 28;
   FAIRE student_id = 1001 JUSQU_À 1040;
      /* Solde du capital au moment du diplôme : 9 500 $ - 48 000 $ */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* TAEG nominal annuel selon la tranche de crédit : 4,0 % - 9,5 % */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Durée de remboursement standard : 120 ou 180 mois */
      SI rand('uniform') < 0.6 ALORS term = 120;
      SINON term = 180;
      /* Frais de dossier en pourcentage du capital : 1,0 % - 2,0 % */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      SORTIE;
   FIN;
EXÉCUTER;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Étape 2 — Profiler la cohorte

Avant de modéliser des prêts individuels, nous examinons la distribution des soldes, des taux et des durées. Cela indique à quoi ressemble un emprunteur *représentatif* — la base de la comparaison directe qui suit.

In [2]:
PROCÉDURE MOYENNES DONNÉES=borrowers n mean MIN MAX maxdec=2;
   VAR balance apr term origination_pct;
   ÉTIQUETTE balance="Solde du capital ($)"
         apr="TAEG (%)"
         term="Durée (mois)"
         origination_pct="Frais de dossier (%)";
EXÉCUTER;

                                                  The MEANS Procedure

 Variable         Label                       N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------
 balance          Solde du capital ($)       40       30771.25    11800.00    47750.00
 apr              TAEG (%)                   40           6.50        4.10        9.10
 term             Durée (mois)               40         144.00      120.00      180.00
 origination_pct  Frais de dossier (%)       40           1.50        1.00        2.00
 -------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 3 — Comparer quatre plans de remboursement pour un emprunteur représentatif

En utilisant un solde représentatif de 27 500 $ sur une durée de 120 mois (10 ans) démarrant en juillet 2026, nous alignons quatre offres réalistes :

- **Fédéral direct non subventionné (Standard)** — un prêt simple à taux fixe de 6,53 %.
- **Refinancement privé (avec frais)** — un taux fixe plus bas de 5,99 %, mais avec un coût d'initialisation de 412,50 $ (`INIT=`).
- **ARM privé à taux variable** — un prêt ajustable à 4,75 % avec un plafond de 1 % par ajustement / 5 % sur la durée de vie (`CAPS=`), un `MAXRATE=` de 9,75 %, un `ADJUSTFREQ=` annuel, et le mot-clé `WORSTCASE`.
- **Rachat de taux 2-1 par l'employeur** — un départ subventionné à 4,50 % qui, selon l'échéancier annoncé, passe via `BDRATES=` à 5,50 % en année 2 puis à 6,50 % ensuite.

L'instruction `COMPARE` demande la vue croisée des prêts à 36, 60 et 120 mois, avec un `TAXRATE=` de 22 % et un taux de rendement minimum attractif de 4 % (`MARR=`) ; `OUTSUM=` et `OUTCOMP=` capturent le récapitulatif par prêt et les lignes de comparaison. Le listing ci-dessous indique, pour chaque plan et chaque point de contrôle, l'**intérêt cumulé payé, le capital cumulé remboursé et le solde restant dû** — une vue claire de la progression de l'amortissement entre les candidats.

> **Remarque sur les ajustements de taux.** `PROC LOAN` de Jenner amortit chaque plan à son taux **de départ** annoncé, de sorte que les réglages `CAPS`/`MAXRATE`/`WORSTCASE` de l'ARM et les paliers `BDRATES` du rachat de taux sont repris dans le listing comme conditions du prêt mais ne sont **pas** appliqués au calcul de la mensualité — les chiffres de l'ARM et du rachat de taux ci-dessous reflètent leurs taux de départ de 4,75 % et 4,50 % maintenus constants sur toute la durée. Considérez ces deux totaux comme des coûts optimistes (taux de départ), et non comme des plafonds de pire cas.

In [3]:
PROCÉDURE loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         ÉTIQUETTE='Fédéral direct non subventionné (Standard)';

   fixed rate=5.99 init=412.50
         ÉTIQUETTE='Refinancement privé (avec frais)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       ÉTIQUETTE='ARM privé à taux variable (pire cas)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           ÉTIQUETTE='Rachat de taux employeur 2-1 puis 6,5 %';

   COMPARER at=(36 60 120) TOUT taxrate=22 marr=4 OUTCOMP=plan_cmp;
EXÉCUTER;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Fédéral direct non subventionné (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Refinancement privé (avec frais)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: ARM privé à taux variable (pire cas)
    Loan Type:                   Adjustable Rate
    Amount (Princi


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Étape 4 — Réexécuter seul le plan fixe recommandé

Pour l'emprunteur qui privilégie la certitude du paiement, le plan standard fédéral à taux fixe est l'option par défaut la plus sûre. Nous le réexécutons isolément pour confirmer son économie principale : la même mensualité de **312,68 $**, un total payé de **37 521,22 $**, et un intérêt total de **10 021,22 $** observés dans la comparaison à quatre, désormais présentés comme un récapitulatif de prêt autonome.

In [4]:
PROCÉDURE loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed ÉTIQUETTE='Fédéral direct non subventionné (Standard)';
EXÉCUTER;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Fédéral direct non subventionné (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Fédéral direct non subventionné (Standard)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4     


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interprétation des résultats

Les quatre plans amortissent entièrement le même capital de 27 500 $ sur 120 mois (le solde restant dû de chaque plan atteint 0,00 $ au mois 120 dans le tableau `COMPARE`), donc la décision repose sur la **mensualité** et l'**intérêt total au taux annoncé** :

| Plan | Taux | Mensualité | Intérêt total | Remarques |
|------|------|---------|----------------|-------|
| Fédéral direct standard | 6,53 % | 312,68 $ | 10 021,22 $ | Sans frais ; protections de l'emprunteur les plus fortes |
| Refinancement privé | 5,99 % | 305,17 $ | 9 120,20 $ | Frais de dossier de 412,50 $ |
| ARM privé à taux variable | 4,75 % (départ) | 288,33 $ | 7 099,76 $ | Le taux peut augmenter ; le chiffre est le coût au taux de départ |
| Rachat de taux employeur 2-1 | 4,50 % (départ) | 285,01 $ | 6 700,67 $ | Dépend de la continuité de l'emploi |

- Le plan **fédéral standard** est le plus coûteux en intérêts (10 021,22 $) mais offre une mensualité fixe et prévisible de 312,68 $ sans frais.
- Le **refinancement privé** réduit le taux à 5,99 % (9 120,20 $ d'intérêts, soit 901 $ de moins que le plan fédéral) mais facture des frais de dossier de 412,50 $, de sorte que son avantage net par rapport au plan fédéral est d'environ 488 $ d'intérêts moins les frais — significatif seulement si le prêt dure assez longtemps pour ne pas être refinancé entre-temps.
- L'**ARM** et le **rachat de taux** affichent ici les intérêts les plus bas (7 099,76 $ et 6 700,67 $) uniquement parce que leurs taux **de départ** sont bien inférieurs aux offres fixes. Comme indiqué à l'étape 3, Jenner maintient ces taux de départ constants, donc ce sont des chiffres optimistes : un véritable ARM qui s'ajuste à la hausse — ou un rachat de taux qui perd sa subvention employeur — atteindrait un montant plus élevé, et la mensualité n'est pas garantie.

Le tableau `COMPARE` précise cela en montrant la vitesse d'amortissement de chaque plan. À 36 mois, le plan fédéral a payé 4 792,27 $ d'intérêts et remboursé 6 464,10 $ de capital (solde 21 035,90 $), tandis que le rachat de taux n'a payé que 3 263,97 $ d'intérêts et remboursé 6 996,24 $ de capital (solde 20 503,76 $) — les plans à taux plus bas coûtent à la fois moins d'intérêts et réduisent le capital plus rapidement au cours des trois premières années. Pour un diplômé averse au risque, la certitude du taux du plan standard fédéral justifie souvent son intérêt plus élevé ; pour un emprunteur confiant dans la stabilité et la durabilité de son emploi, le départ subventionné du rachat de taux offre les économies les plus importantes parmi les options qui fixent réellement leur taux.